# ══════════════════════════════════════════════════════════════
#  WEEK 12  |  BONUS  |  SELF-HEALING ETL PIPELINE
# ══════════════════════════════════════════════════════════════
#
#  LEARNING OBJECTIVES
#  ───────────────────
#  1. Build a multi-stage ETL pipeline that never crashes -- it logs,
#     recovers, and explains every error in plain English
#  2. Integrate an LLM at 4 key points in the pipeline lifecycle
#  3. Use Parquet as the standard output format (Silver layer)
#  4. Query Silver layer data with DuckDB (local Athena equivalent)
#  5. Generate human-readable audit logs via LLM
#
#  ARCHITECTURE (simulation mode -- no cloud required)
#  ────────────────────────────────────────────────────
#
#    Raw CSV/JSON files (local)
#          |
#          v
#    Stage 1: LLM Pre-Validation -- inspect sample, flag anomalies
#          |
#          v
#    Bronze layer (local folder -- simulates MinIO/S3)
#          |
#          v
#    Stage 2: ETL -- detect schema drift, LLM semantic mapping, clean
#          |
#          v
#    Silver layer (Parquet files -- simulates MinIO/S3)
#          |
#          v
#    Stage 3: DuckDB query layer (simulates AWS Athena)
#          |
#          v
#    Stage 4: NL Audit Log (JSON file -- LLM-generated entries)
#
#  SETUP
#  ─────
#  pip install langchain-groq langchain-core pydantic python-dotenv
#             pandas pyarrow duckdb
#
#  Optional (for LLM stages): copy nl_sql_app/.env.example to .env
#  and add your Groq API key. All LLM stages gracefully skip if key is absent.
#
#  TIME:  ~90 minutes
#
#  ─────────────────────────────────────────────────────────────
#  ARCHITECTURE DECISION
#  ─────────────────────
#  Choosing between: AWS Glue vs Python scripts vs Spark for ETL.
#  Rule of thumb: use Python scripts when data fits in memory (<10GB).
#  Move to Spark when you need distributed processing across a cluster.
#  Use Glue when you need serverless ETL with AWS-native scheduling.
#
# ══════════════════════════════════════════════════════════════

In [ ]:
import os
import json
import pandas as pd
import duckdb
import tempfile
import traceback
from datetime import datetime
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

# Pipeline output directories (simulate Bronze/Silver layers)
BRONZE_DIR = os.path.join(tempfile.gettempdir(), 'etl_pipeline', 'bronze')
SILVER_DIR = os.path.join(tempfile.gettempdir(), 'etl_pipeline', 'silver')
AUDIT_FILE = os.path.join(tempfile.gettempdir(), 'etl_pipeline', 'audit_log.json')

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

print('=== Self-Healing ETL Pipeline ===')
print(f'Bronze dir : {BRONZE_DIR}')
print(f'Silver dir : {SILVER_DIR}')
print(f'Audit file : {AUDIT_FILE}')
print(f'Groq key   : {"set" if GROQ_API_KEY else "not set -- LLM stages will use mock responses"}')

# ══════════════════════════════════════════════════════════════
#  CONCEPT 1 -- THE FOUR LLM INTEGRATION STAGES
# ══════════════════════════════════════════════════════════════
#
#  Traditional ETL: Extract -> Transform -> Load
#  Self-healing ETL adds LLM intelligence at 4 points:
#
#  STAGE 1 -- PRE-VALIDATION (before Bronze)
#    LLM inspects a data sample and flags anomalies BEFORE the
#    pipeline processes anything. Catches bad data at the source.
#    Returns: quality_score (0-10), anomalies list, recommendation.
#
#  STAGE 2 -- SEMANTIC SCHEMA MAPPING (Bronze -> Silver)
#    When column names change (schema drift), instead of crashing,
#    the LLM maps the new names to the expected ones.
#    Example: 'customer_id' and 'user_id' are probably the same thing.
#
#  STAGE 3 -- ERROR REMEDIATION (any stage)
#    If any stage fails, the traceback is sent to the LLM which
#    explains the error in plain English and suggests a fix.
#    The pipeline logs the explanation instead of just re-raising.
#
#  STAGE 4 -- NL AUDIT LOGGING (after each stage)
#    Instead of 'processed 1000 rows', the LLM writes:
#    'Successfully processed 1,000 HR records. 38 were quarantined
#     due to missing salary fields.'
#    This goes into a JSON audit file alongside standard logs.
#
#  EXAMPLE -- Pydantic models for each stage ───────────────────

In [ ]:
# ── Pydantic models for pipeline data contracts ───────────────

class ValidationReport(BaseModel):
    """Stage 1: pre-validation result."""
    schema_match:   bool
    anomalies:      list
    quality_score:  int   # 0-10
    recommendation: str


class SchemaMappingResult(BaseModel):
    """Stage 2: schema drift resolution."""
    drift_detected: bool
    column_mapping: dict   # {'new_name': 'expected_name', ...}
    confidence:     str    # 'high' | 'medium' | 'low'


class RemediationReport(BaseModel):
    """Stage 3: error explanation."""
    explanation:   str
    suggested_fix: str


class AuditEntry(BaseModel):
    """Stage 4: one audit log entry."""
    stage:     str
    nl_entry:  str
    timestamp: str


print('Pydantic models loaded:')
print('  ValidationReport  -- Stage 1 pre-validation')
print('  SchemaMappingResult -- Stage 2 schema drift')
print('  RemediationReport -- Stage 3 error explanation')
print('  AuditEntry        -- Stage 4 NL audit log')

# ══════════════════════════════════════════════════════════════
#  CONCEPT 2 -- THE SELF-HEALING PATTERN
# ══════════════════════════════════════════════════════════════
#
#  A self-healing pipeline NEVER crashes the entire run because of
#  a single bad record or unexpected column name.
#
#  Key design rules:
#
#  1. QUARANTINE bad records instead of dropping them
#     Bad rows go to a 'quarantine' file, not to the trash.
#     You can review and reprocess them later.
#
#  2. EXPLAIN failures before continuing
#     Every except block calls the LLM to explain the traceback.
#     Engineers get plain English, not cryptic stack traces.
#
#  3. LOG every run with a structured audit trail
#     {stage, records_in, records_out, errors, duration_s, nl_entry}
#     Both for machines (JSON) and humans (NL sentence).
#
#  4. DEGRADE GRACEFULLY when LLM is unavailable
#     If GROQ_API_KEY is not set, use rule-based fallbacks.
#     The pipeline still runs -- just without AI-enhanced logging.
#
#  EXAMPLE -- the self-healing try/except pattern ──────────────

In [ ]:
# ── Self-healing wrapper pattern ─────────────────────────────

def get_llm():
    """Return a Groq LLM if key is set, otherwise None."""
    if not GROQ_API_KEY:
        return None
    from langchain_groq import ChatGroq
    return ChatGroq(model='llama-3.1-70b-versatile', api_key=GROQ_API_KEY, temperature=0)


def explain_error(tb_text: str, stage: str) -> RemediationReport:
    """Ask the LLM to explain a traceback. Falls back to a template."""
    llm = get_llm()
    if not llm:
        return RemediationReport(
            explanation=f'Stage {stage} failed. Check the traceback above for details.',
            suggested_fix='Review the traceback and fix the root cause in the source data or code.'
        )
    from langchain_core.messages import HumanMessage
    prompt = (
        f'You are a senior data engineer. Pipeline stage {stage!r} failed.\n\n'
        f'Traceback:\n{tb_text}\n\n'
        'Respond ONLY with JSON: {"explanation": "...", "suggested_fix": "..."}'
    )
    raw = llm.invoke([HumanMessage(content=prompt)]).content
    s = raw.find('{'); e = raw.rfind('}') + 1
    return RemediationReport(**json.loads(raw[s:e]))


# ── Demo: catch + explain an error ───────────────────────────
try:
    result = {'data': [1, 2, 3]}['missing_key']  # intentional KeyError
except Exception:
    tb = traceback.format_exc()
    report = explain_error(tb, 'extract')
    print('Error explanation:')
    print(f'  {report.explanation}')
    print(f'  Suggested fix: {report.suggested_fix}')

# ══════════════════════════════════════════════════════════════
#  CONCEPT 3 -- BRONZE / SILVER / GOLD DATA LAKE ARCHITECTURE
# ══════════════════════════════════════════════════════════════
#
#  Data lakes organize files into three quality tiers:
#
#  BRONZE (raw layer)
#    Exact copy of the source data, never modified.
#    Retained indefinitely for audit and reprocessing.
#    Format: original CSV, JSON, or Parquet as received.
#    Production: s3://bucket/bronze/YYYY/MM/DD/filename
#
#  SILVER (clean layer)
#    Validated, schema-aligned, deduplicated records.
#    Bad rows are quarantined, not deleted.
#    Format: Parquet (compressed, typed, queryable).
#    Production: s3://bucket/silver/entity/YYYY/MM/
#
#  GOLD (business layer) -- not built in this project
#    Aggregated, business-ready tables.
#    Joins, groupBys, and metrics applied.
#    Used by BI tools (Grafana, Tableau, Power BI).
#    Production: s3://bucket/gold/report_name/
#
#  Simulation in this project:
#    Bronze: tempdir/etl_pipeline/bronze/  (raw CSV copy)
#    Silver: tempdir/etl_pipeline/silver/  (clean Parquet)
#    Gold:   DuckDB queries on Silver files  (on demand)
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
# ── Bronze/Silver layer demo ──────────────────────────────────
import os
import pandas as pd

# Show the directory structure of the simulated data lake
print('Data lake directory structure (simulated):')
print(f'  Bronze: {BRONZE_DIR}')
print(f'  Silver: {SILVER_DIR}')
print()

# Write a raw file to Bronze (exact copy, no transformation)
sample_raw = pd.DataFrame({
    'employee_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Carol'],
    'salary': [95000, 72000, 85000],
})
bronze_path = os.path.join(BRONZE_DIR, 'demo_raw.parquet')
sample_raw.to_parquet(bronze_path, index=False)

# Write a cleaned file to Silver (validated, typed)
sample_clean = sample_raw.copy()
silver_demo_path = os.path.join(SILVER_DIR, 'demo_clean.parquet')
sample_clean.to_parquet(silver_demo_path, index=False)

print('Bronze layer (raw):')
print(f'  {bronze_path}')
print(f'  Rows: {len(pd.read_parquet(bronze_path))}')
print()
print('Silver layer (clean):')
print(f'  {silver_demo_path}')
print(f'  Rows: {len(pd.read_parquet(silver_demo_path))}')
print()
print('Rule: Bronze is immutable. Silver may have fewer rows (quarantine applied).')

# ══════════════════════════════════════════════════════════════
#  TASK 1 -- STAGE 1: LLM PRE-VALIDATION
# ══════════════════════════════════════════════════════════════
#
#  Write validate_raw_data(df: pd.DataFrame, sample_size: int = 20)
#  -> ValidationReport that:
#
#    1. Takes a sample of the DataFrame (first sample_size rows)
#    2. Builds a prompt asking the LLM to check against the schema:
#         employee_id (int), name (str), department (str),
#         salary (float > 0), email (valid email with @),
#         hire_date (ISO date string)
#    3. If GROQ_API_KEY is set: call Groq, parse JSON into ValidationReport
#    4. If not set: inspect the sample manually (check for None, negative
#       salary, missing @) and return a ValidationReport with those findings
#    5. Print the report and return it
#
#  Then call validate_raw_data(raw_hr_data) on the starting data.
#
#  Expected output:
#    Pre-validation result:
#    Quality score : 4/10
#    Schema match  : False
#    Anomalies     : 3 found
#      - salary -500 is invalid (negative)
#      - email 'not-valid' missing @
#      - name is None
#    Recommendation: Fix invalid salary, email, and null name before ingesting.
#
# --- starting data ---

In [ ]:
import pandas as pd

# Raw HR data with intentional quality issues
raw_hr_data = pd.DataFrame({
    'employee_id': [1, 2, 3, 4, 5, 6],
    'name':        ['Alice', 'Bob', None, 'Dave', 'Eve', 'Frank'],
    'department':  ['Engineering', 'Sales', 'Finance', 'Engineering', 'HR', 'Sales'],
    'salary':      [95000.0, 72000.0, 85000.0, -500.0, 68000.0, 0.0],
    'email':       ['alice@co.com', 'bob@co.com', 'carol@co.com', 'not-valid', 'eve@co.com', 'frank@co.com'],
    'hire_date':   ['2022-03-01', '2021-06-15', '2023-01-10', '2020-09-01', '2024-02-20', '2019-11-05'],
})

# ══════════════════════════════════════════════════════════════
#  TASK 2 -- STAGE 2: DETECT AND MAP SCHEMA DRIFT
# ══════════════════════════════════════════════════════════════
#
#  Write detect_and_map_schema(df: pd.DataFrame, expected_cols: set)
#  -> tuple[pd.DataFrame, SchemaMappingResult] that:
#
#    1. Detect drift: compare set(df.columns) vs expected_cols
#    2. If no drift: return (df, SchemaMappingResult(drift_detected=False,
#       column_mapping={}, confidence='high'))
#    3. If drift detected and GROQ_API_KEY is set:
#       a. Build a prompt asking the LLM to map new column names to
#          expected names (return JSON: {new_name: expected_name, ...})
#       b. Apply the mapping to rename columns in the DataFrame
#       c. Return (renamed_df, SchemaMappingResult(drift_detected=True,
#          column_mapping=mapping, confidence='high'))
#    4. If drift detected and no GROQ key: use simple heuristics
#       (if a new name contains a substring of an expected name, map it)
#       Return with confidence='low'
#
#  Call detect_and_map_schema(drifted_df, EXPECTED_COLS) on the starting data.
#
#  Expected output:
#    Drift detected: True
#    Mapping applied:
#      customer_id -> employee_id
#      full_name   -> name
#      dept        -> department
#    DataFrame columns after mapping: ['employee_id', 'name', 'department', ...]
#
# --- starting data ---

In [ ]:
import pandas as pd

EXPECTED_COLS = {'employee_id', 'name', 'department', 'salary', 'email', 'hire_date'}

# Simulates a source system that renamed its columns
drifted_df = pd.DataFrame({
    'customer_id': [1, 2, 3],      # was: employee_id
    'full_name':   ['Alice', 'Bob', 'Carol'],  # was: name
    'dept':        ['Eng', 'Sales', 'Finance'], # was: department
    'salary':      [95000, 72000, 85000],
    'email':       ['alice@co.com', 'bob@co.com', 'carol@co.com'],
    'hire_date':   ['2022-03-01', '2021-06-15', '2023-01-10'],
})

# ══════════════════════════════════════════════════════════════
#  TASK 3 -- STAGE 3: TRANSFORM AND SAVE TO SILVER LAYER
# ══════════════════════════════════════════════════════════════
#
#  Write transform_to_parquet(df: pd.DataFrame, output_path: str)
#  -> dict that:
#
#    1. Clean the DataFrame:
#       a. Drop rows where salary <= 0
#       b. Drop rows where email does not contain '@'
#       c. Drop rows where name is None
#       d. Quarantine the dropped rows -- save to output_path + '_quarantine.parquet'
#
#    2. Save the clean rows to output_path (Parquet format)
#
#    3. Verify: reload from output_path, confirm row count matches
#
#    4. Return a stats dict:
#       {'records_in': N, 'records_out': M, 'quarantined': Q,
#        'output_path': output_path, 'quarantine_path': ...}
#
#  Call transform_to_parquet(raw_hr_data, silver_path) on the starting data.
#
#  Expected output:
#    Transform stats:
#      Records in       : 6
#      Records out      : 3
#      Quarantined      : 3
#      Silver path      : .../etl_pipeline/silver/hr_employees.parquet
#      Quarantine path  : .../etl_pipeline/silver/hr_employees_quarantine.parquet
#
# --- starting data ---

In [ ]:
import pandas as pd
import os

silver_path = os.path.join(SILVER_DIR, 'hr_employees.parquet')

# Use raw_hr_data from TASK 1 as input
# (raw_hr_data has 3 bad rows: negative salary, missing @, None name)

# ══════════════════════════════════════════════════════════════
#  TASK 4 -- STAGE 4: QUERY THE SILVER LAYER WITH DUCKDB
# ══════════════════════════════════════════════════════════════
#
#  Write query_silver_layer(parquet_path: str, sql: str) -> pd.DataFrame
#  that executes the given SQL against the Parquet file using DuckDB.
#
#  The function must:
#    1. Use duckdb.sql(f"... FROM '{parquet_path}' ...").df()
#    2. Wrap in try/except -- if the query fails, call explain_error()
#       from CONCEPT 2 and raise a ValueError with the explanation
#
#  Then run these two queries and print results:
#    a. Average salary per department (GROUP BY + AVG)
#    b. Employees sorted by salary descending
#
#  Expected output:
#    Average salary by department:
#    department  avg_salary
#    Engineering  95000.0
#    Finance      85000.0
#    HR           68000.0
#
#    Employees by salary:
#    name      department  salary
#    Alice     Engineering  95000.0
#    Carol     Finance      85000.0
#    Eve       HR           68000.0
#
# --- starting data ---

In [ ]:
import duckdb

# Use silver_path from TASK 3
# (points to the clean Parquet file in SILVER_DIR)

# ══════════════════════════════════════════════════════════════
#  TASK 5 -- WIRE ALL STAGES INTO run_pipeline()
# ══════════════════════════════════════════════════════════════
#
#  Write run_pipeline(input_csv_path: str) -> list that:
#
#  1. Extract: read the CSV into a DataFrame, save raw copy to BRONZE_DIR
#
#  2. Stage 1 - Pre-Validation:
#     Call validate_raw_data(df). If quality_score < 5, log a warning
#     but continue (do not abort -- self-healing means we proceed).
#
#  3. Stage 2 - Schema mapping:
#     Call detect_and_map_schema(df, EXPECTED_COLS). Apply the result.
#
#  4. Stage 3 - Transform:
#     Call transform_to_parquet(df, silver_path). Collect stats.
#
#  5. Stage 4 - Audit logging:
#     For each completed stage, generate a NL audit entry:
#       - If GROQ_API_KEY set: call LLM with stage stats dict
#       - If not set: build a template sentence from the stats
#     Write all entries to AUDIT_FILE as a JSON list.
#
#  6. Return the list of AuditEntry objects
#
#  Then:
#    a. Write pipeline_csv to a temp file and call run_pipeline()
#    b. Print all audit entries
#    c. Run one DuckDB query on the Silver layer output
#
#  Expected output:
#    Pipeline completed.
#
#    Audit log:
#      [extract]   Read 8 records from employees_raw.csv.
#      [validate]  Quality score 4/10. 3 anomalies found.
#      [transform] 5 records saved to Silver. 3 quarantined.
#      [query]     Average salary: Engineering=$95,000, Sales=$72,000.
#    Audit written to: .../audit_log.json
#
# --- starting data ---

In [ ]:
import os
import pandas as pd
import tempfile

# Raw input data -- simulates a CSV delivered by the upstream HR system
pipeline_data = {
    'employee_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'name':        ['Alice', 'Bob', None, 'Dave', 'Eve', 'Frank', 'Grace', 'Hank'],
    'department':  ['Engineering', 'Sales', 'Finance', 'Engineering', 'HR', 'Sales', 'Engineering', 'Finance'],
    'salary':      [95000.0, 72000.0, 85000.0, -500.0, 68000.0, 0.0, 91000.0, 78000.0],
    'email':       ['alice@co.com', 'bob@co.com', 'carol@co.com', 'not-valid',
                    'eve@co.com', 'frank@co.com', 'grace@co.com', 'hank@co.com'],
    'hire_date':   ['2022-03-01', '2021-06-15', '2023-01-10', '2020-09-01',
                    '2024-02-20', '2019-11-05', '2023-08-12', '2022-07-30'],
}

# Write to a temp CSV to simulate an upstream file delivery
pipeline_csv = os.path.join(tempfile.gettempdir(), 'employees_raw.csv')
pd.DataFrame(pipeline_data).to_csv(pipeline_csv, index=False)
print(f'Input CSV written: {pipeline_csv}')